# Modulo 2: Supervised Learning e Ottimizzazione (CORRETTO)
## Predizione del PEGI dei Videogiochi di Steam — Solo Tag

### Fix metodologici rispetto alla versione precedente:
1. **Train/Test Split stratificato (80/20)** prima di feature selection e bilanciamento
2. **Feature Selection solo sul train set** — nessun data leakage
3. **class_weight='balanced'** al posto di BorderlineSMOTE su feature binarie
4. **Tuning Optuna per TUTTI i 4 modelli** (inclusi CatBoost ed ExtraTrees)
5. **Valutazione finale SOLO sul test set** — metriche oneste
6. **Mappatura PEGI estesa** — recupero ~460 campioni con rating non standard

### Flusso del progetto:
1. **Feature Engineering** – Caricamento dati, encoding tag, mappatura PEGI estesa
2. **Setup** – Train/test split, feature selection (solo su train), configurazione
3. **Ottimizzazione XGBoost** – Optuna 30 trial con class_weight bilanciato
4. **Ottimizzazione LightGBM** – Optuna 20 trial con class_weight bilanciato
5. **Ottimizzazione CatBoost** – Optuna 20 trial con auto_class_weights
6. **Ottimizzazione ExtraTrees** – Optuna 20 trial con class_weight bilanciato
7. **Ensemble Stacking** – 4 base learner ottimizzati + meta-learner LogisticRegression
8. **Valutazione Finale** – Metriche sul TEST SET, confusion matrix, ROC, SHAP

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import MultiLabelBinarizer
import ast
import warnings
warnings.filterwarnings('ignore')

print('[INFO] Sezione 1: Feature Engineering')
print('=' * 70)

## Sezione 1: Feature Engineering e Caricamento Dati

In [ ]:
# Caricamento dataset (supporto locale e Colab)
csv_path = None
for candidate in ['/content/for_EDA.csv', 'for_EDA.csv']:
    if os.path.exists(candidate):
        csv_path = candidate
        break

if csv_path is None:
    raise FileNotFoundError('[ERROR] for_EDA.csv non trovato!')

df = pd.read_csv(csv_path)
print(f'[INFO] Dataset caricato: {df.shape[0]} righe x {df.shape[1]} colonne')
print(f'[INFO] Colonne: {list(df.columns[:10])}...')

In [ ]:
# Ispeziona struttura del dataset
print('[INFO] Struttura del dataset for_EDA.csv:')
print(f'\nColonne disponibili ({len(df.columns)}):')
print(df.columns.tolist())
print(f'\nTipi di dati:')
print(df.dtypes)
print(f'\nPrime righe:')
df.head()

In [ ]:
# Multi-Hot Encoding sui tag (se la colonna 'tags' esiste)
if 'tags' in df.columns:
    def clean_tags(t):
        if pd.isna(t) or not isinstance(t, str):
            return []
        t = t.strip()
        if t.startswith('[') and t.endswith(']'):
            try:
                return ast.literal_eval(t)
            except (ValueError, SyntaxError):
                pass
        return [x.strip() for x in t.split(',') if x.strip()]

    df['tags_cleaned'] = df['tags'].apply(clean_tags)
    mlb = MultiLabelBinarizer()
    tags_enc = mlb.fit_transform(df['tags_cleaned'])
    tags_cols = [f"Tag_{g.replace(' ','_').replace('-','_').replace('/','_')}" for g in mlb.classes_]
    tags_df = pd.DataFrame(tags_enc, columns=tags_cols, index=df.index)
    df_encoded = tags_df.copy()

    print(f'[INFO] Tag unici codificati: {len(mlb.classes_)}')
else:
    print('[INFO] Colonna "tags" non trovata, impossibile eseguire il training basato solo sui tag.')
    raise ValueError('Impossibile trovare colonna tags')

print(f'[INFO] Shape dopo multi-hot encoding tag: {df_encoded.shape}')

In [ ]:
# Mappatura target PEGI -> classi [0-4]
# FIX: Mappatura ESTESA per recuperare campioni con rating non standard
pegi_col = None
for candidate in ['pegi', 'required_age', 'age_rating', 'rating']:
    if candidate in df.columns:
        pegi_col = candidate
        break

if pegi_col is None:
    print('[WARNING] Nessuna colonna PEGI trovata!')
    print(f'Colonne disponibili: {df.columns.tolist()}')
    raise ValueError('Impossibile trovare colonna target per PEGI')

# Mappatura estesa: rating non standard -> classe PEGI piu' vicina
pegi_mapping = {
    3: 0, 4: 0,      # PEGI 3 (include rating 4)
    7: 1, 6: 1,      # PEGI 7 (include rating 6)
    12: 2, 11: 2,    # PEGI 12 (include rating 11)
    16: 3, 15: 3,    # PEGI 16 (include rating 15)
    18: 4             # PEGI 18
}

pegi_class_names = {0: 'PEGI 3', 1: 'PEGI 7', 2: 'PEGI 12', 3: 'PEGI 16', 4: 'PEGI 18'}

df_encoded['target_class'] = df[pegi_col].map(pegi_mapping)

# Conta campioni recuperati con mappatura estesa
n_before = df[pegi_col].isin([3, 7, 12, 16, 18]).sum()
n_after = df[pegi_col].map(pegi_mapping).notna().sum()
print(f'[INFO] Campioni con rating PEGI standard: {n_before}')
print(f'[INFO] Campioni con mappatura estesa: {n_after} (+{n_after - n_before} recuperati)')

df_encoded = df_encoded.dropna(subset=['target_class']).copy()
df_encoded['target_class'] = df_encoded['target_class'].astype(int)
df_final = df_encoded

print(f'\n[INFO] Colonna target utilizzata: "{pegi_col}"')
print('[INFO] Distribuzione classi target:')
for ci in sorted(pegi_class_names.keys()):
    count = (df_final['target_class'] == ci).sum()
    pct = count / len(df_final) * 100 if len(df_final) > 0 else 0
    print(f'   {pegi_class_names[ci]} (Classe {ci}): {count:5d} campioni ({pct:5.1f}%)')

print(f'\n[INFO] Shape dataset finale: {df_final.shape}')

## Sezione 2: Setup - Train/Test Split e Feature Selection

> **FIX CRITICO**: Split train/test PRIMA di feature selection per evitare data leakage.
> La feature selection viene fatta SOLO sul train set e poi applicata (transform) al test set.

In [ ]:
!pip install -q optuna xgboost lightgbm catboost shap scikit-learn

In [ ]:
import joblib
import optuna
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import ExtraTreesClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight

optuna.logging.set_verbosity(optuna.logging.WARNING)

print('[INFO] Sezione 2: Setup e configurazione')
print('=' * 70)

In [ ]:
# ============================================================
# FIX CRITICO: Split TRAIN/TEST stratificato PRIMA di tutto
# ============================================================
X = df_final.drop(columns=['target_class'])
y = df_final['target_class'].values

# I tag sono gia' codificati come colonne binarie
X_tags = X.fillna(0).astype(int)

# Split 80/20 stratificato
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_tags, y, test_size=0.2, random_state=42, stratify=y
)

print(f'[INFO] Shape X tags totale: {X_tags.shape}')
print(f'[INFO] Train set: {X_train_full.shape[0]} campioni')
print(f'[INFO] Test set:  {X_test_full.shape[0]} campioni')
print(f'\n[INFO] Distribuzione nel train:')
for ci in sorted(pegi_class_names.keys()):
    count = (y_train == ci).sum()
    pct = count / len(y_train) * 100
    print(f'   {pegi_class_names[ci]}: {count} ({pct:.1f}%)')
print(f'\n[INFO] Distribuzione nel test:')
for ci in sorted(pegi_class_names.keys()):
    count = (y_test == ci).sum()
    pct = count / len(y_test) * 100
    print(f'   {pegi_class_names[ci]}: {count} ({pct:.1f}%)')

In [ ]:
# ============================================================
# Feature Selection con Mutual Information — SOLO SUL TRAIN SET
# ============================================================
K_FEATURES = min(200, X_train_full.shape[1])
print(f'[INFO] Selezione top-{K_FEATURES} feature via mutual_info_classif (solo su train)...')

selector = SelectKBest(score_func=mutual_info_classif, k=K_FEATURES)
X_train_sel_arr = selector.fit_transform(X_train_full, y_train)
sel_names = X_train_full.columns[selector.get_support()].tolist()
X_train_sel = pd.DataFrame(X_train_sel_arr, columns=sel_names)

# Applica la STESSA trasformazione al test set (nessun re-fitting!)
X_test_sel = pd.DataFrame(
    selector.transform(X_test_full),
    columns=sel_names
)

print(f'[INFO] Feature selezionate: {X_train_sel.shape[1]}')
print(f'[INFO] Train selezionato: {X_train_sel.shape}')
print(f'[INFO] Test selezionato:  {X_test_sel.shape}')

joblib.dump(selector, 'feature_selector.pkl')
joblib.dump(sel_names, 'selected_features.pkl')

In [ ]:
# Configurazione validazione e bilanciamento
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# ============================================================
# FIX: Wrapper XGBoost con class_weight automatico
# XGBoost non supporta class_weight nativamente, quindi calcoliamo
# sample_weight bilanciati automaticamente in fit()
# ============================================================
class BalancedXGBClassifier(xgb.XGBClassifier):
    """XGBClassifier che applica automaticamente sample_weight bilanciati."""
    def fit(self, X, y, **kwargs):
        if 'sample_weight' not in kwargs:
            kwargs['sample_weight'] = compute_sample_weight('balanced', y)
        return super().fit(X, y, **kwargs)

print(f'[INFO] StratifiedKFold: {N_SPLITS} fold')
print(f'[INFO] Bilanciamento: class_weight="balanced" (NO SMOTE su feature binarie)')
print(f'[INFO] Setup completato!\n')

## Sezione 3a: Ottimizzazione XGBoost con Optuna (30 trial)

In [ ]:
print('[INFO] Sezione 3a: Ottimizzazione XGBoost')
print('=' * 70)

def objective_xgb(trial):
    params = {
        'objective': 'multi:softprob',
        'num_class': 5,
        'eval_metric': 'mlogloss',
        'random_state': 42,
        'n_jobs': -1,
        # Struttura alberi
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'n_estimators': trial.suggest_int('n_estimators', 150, 500),
        # Learning
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        # Campionamento stocastico
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.5, 1.0),
        # Regolarizzazione
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
    }

    fold_scores = []
    for tr_idx, vl_idx in skf.split(X_train_sel, y_train):
        Xtr, Xvl = X_train_sel.iloc[tr_idx], X_train_sel.iloc[vl_idx]
        ytr, yvl = y_train[tr_idx], y_train[vl_idx]

        # FIX: class_weight via sample_weight invece di SMOTE
        sw = compute_sample_weight('balanced', ytr)

        model = xgb.XGBClassifier(**params)
        model.fit(Xtr, ytr, sample_weight=sw, verbose=False)
        fold_scores.append(f1_score(yvl, model.predict(Xvl), average='macro'))

    return np.mean(fold_scores)

study_xgb = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
)
study_xgb.optimize(objective_xgb, n_trials=30, show_progress_bar=True)

print(f'\n[XGBoost] Miglior F1 Macro (CV su train): {study_xgb.best_value:.4f}')
print('Migliori parametri XGBoost:')
for k, v in study_xgb.best_params.items():
    print(f'   {k}: {v}')

## Sezione 3b: Ottimizzazione LightGBM con Optuna (20 trial)

In [ ]:
print('\n[INFO] Sezione 3b: Ottimizzazione LightGBM')
print('=' * 70)

def objective_lgb(trial):
    params = {
        'objective': 'multiclass',
        'num_class': 5,
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1,
        'class_weight': 'balanced',  # FIX: bilanciamento nativo
        'n_estimators': trial.suggest_int('n_estimators', 150, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 60),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
    }

    fold_scores = []
    for tr_idx, vl_idx in skf.split(X_train_sel, y_train):
        Xtr, Xvl = X_train_sel.iloc[tr_idx], X_train_sel.iloc[vl_idx]
        ytr, yvl = y_train[tr_idx], y_train[vl_idx]

        model = lgb.LGBMClassifier(**params)
        model.fit(Xtr, ytr)
        fold_scores.append(f1_score(yvl, model.predict(Xvl), average='macro'))

    return np.mean(fold_scores)

study_lgb = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
)
study_lgb.optimize(objective_lgb, n_trials=20, show_progress_bar=True)

print(f'\n[LightGBM] Miglior F1 Macro (CV su train): {study_lgb.best_value:.4f}')
print('Migliori parametri LightGBM:')
for k, v in study_lgb.best_params.items():
    print(f'   {k}: {v}')

## Sezione 3c: Ottimizzazione CatBoost con Optuna (20 trial)

> **FIX**: CatBoost ora ottimizzato con Optuna (prima usava iperparametri fissi)

In [ ]:
print('\n[INFO] Sezione 3c: Ottimizzazione CatBoost')
print('=' * 70)

def objective_cat(trial):
    params = {
        'loss_function': 'MultiClass',
        'random_state': 42,
        'thread_count': -1,
        'verbose': 0,
        'auto_class_weights': 'Balanced',  # FIX: bilanciamento nativo
        'iterations': trial.suggest_int('iterations', 200, 600),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength': trial.suggest_float('random_strength', 1e-3, 10.0, log=True),
    }

    fold_scores = []
    for tr_idx, vl_idx in skf.split(X_train_sel, y_train):
        Xtr, Xvl = X_train_sel.iloc[tr_idx], X_train_sel.iloc[vl_idx]
        ytr, yvl = y_train[tr_idx], y_train[vl_idx]

        model = CatBoostClassifier(**params)
        model.fit(Xtr, ytr)
        fold_scores.append(f1_score(yvl, model.predict(Xvl), average='macro'))

    return np.mean(fold_scores)

study_cat = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
)
study_cat.optimize(objective_cat, n_trials=20, show_progress_bar=True)

print(f'\n[CatBoost] Miglior F1 Macro (CV su train): {study_cat.best_value:.4f}')
print('Migliori parametri CatBoost:')
for k, v in study_cat.best_params.items():
    print(f'   {k}: {v}')

## Sezione 3d: Ottimizzazione ExtraTrees con Optuna (20 trial)

> **FIX**: ExtraTrees ora ottimizzato con Optuna (prima usava iperparametri fissi)

In [ ]:
print('\n[INFO] Sezione 3d: Ottimizzazione ExtraTrees')
print('=' * 70)

def objective_et(trial):
    params = {
        'random_state': 42,
        'n_jobs': -1,
        'class_weight': 'balanced',  # Bilanciamento nativo
        'n_estimators': trial.suggest_int('n_estimators', 200, 600),
        'max_depth': trial.suggest_int('max_depth', 10, 50),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
    }

    fold_scores = []
    for tr_idx, vl_idx in skf.split(X_train_sel, y_train):
        Xtr, Xvl = X_train_sel.iloc[tr_idx], X_train_sel.iloc[vl_idx]
        ytr, yvl = y_train[tr_idx], y_train[vl_idx]

        model = ExtraTreesClassifier(**params)
        model.fit(Xtr, ytr)
        fold_scores.append(f1_score(yvl, model.predict(Xvl), average='macro'))

    return np.mean(fold_scores)

study_et = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
)
study_et.optimize(objective_et, n_trials=20, show_progress_bar=True)

print(f'\n[ExtraTrees] Miglior F1 Macro (CV su train): {study_et.best_value:.4f}')
print('Migliori parametri ExtraTrees:')
for k, v in study_et.best_params.items():
    print(f'   {k}: {v}')

In [ ]:
# Riepilogo tuning Optuna
print('\n' + '=' * 70)
print('RIEPILOGO TUNING OPTUNA (F1 Macro CV su train set)')
print('=' * 70)
print(f'  XGBoost:    {study_xgb.best_value:.4f}')
print(f'  LightGBM:   {study_lgb.best_value:.4f}')
print(f'  CatBoost:   {study_cat.best_value:.4f}')
print(f'  ExtraTrees: {study_et.best_value:.4f}')
print('=' * 70)
print('Nota: queste sono stime oneste (CV, no leakage)')

## Sezione 4: Addestramento Ensemble - StackingClassifier

In [ ]:
print('\n[INFO] Sezione 4: Addestramento Ensemble Stacking')
print('=' * 70)

# Costruzione base learner con parametri ottimizzati da Optuna
best_xgb_p = {
    **study_xgb.best_params,
    'objective': 'multi:softprob',
    'num_class': 5,
    'eval_metric': 'mlogloss',
    'random_state': 42,
    'n_jobs': -1
}

best_lgb_p = {
    **study_lgb.best_params,
    'objective': 'multiclass',
    'num_class': 5,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
    'class_weight': 'balanced'
}

best_cat_p = {
    **study_cat.best_params,
    'loss_function': 'MultiClass',
    'random_state': 42,
    'thread_count': -1,
    'verbose': 0,
    'auto_class_weights': 'Balanced'
}

best_et_p = {
    **study_et.best_params,
    'random_state': 42,
    'n_jobs': -1,
    'class_weight': 'balanced'
}

# FIX: Usa BalancedXGBClassifier per gestire sample_weight automaticamente
xgb_final = BalancedXGBClassifier(**best_xgb_p)
lgb_final = lgb.LGBMClassifier(**best_lgb_p)
cat_final = CatBoostClassifier(**best_cat_p)
et_final = ExtraTreesClassifier(**best_et_p)

print('[INFO] Base learner creati (TUTTI ottimizzati con Optuna):')
print('   - XGBoost (ottimizzato, class_weight via sample_weight)')
print('   - LightGBM (ottimizzato, class_weight=balanced)')
print('   - CatBoost (ottimizzato, auto_class_weights=Balanced)')
print('   - ExtraTrees (ottimizzato, class_weight=balanced)')

In [ ]:
# Meta-learner con class_weight bilanciato
meta_learner = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000, C=1.0, multi_class='multinomial',
        random_state=42, class_weight='balanced'
    )
)

stacking_clf = StackingClassifier(
    estimators=[
        ('xgb', xgb_final),
        ('lgb', lgb_final),
        ('cat', cat_final),
        ('et', et_final),
    ],
    final_estimator=meta_learner,
    cv=5,
    stack_method='predict_proba',
    n_jobs=-1,
    passthrough=False
)

print('[INFO] StackingClassifier configurato (4 base + LogReg meta-learner)')

# ============================================================
# FIX CRITICO: Addestramento SOLO sul TRAIN set (no SMOTE!)
# ============================================================
print('[INFO] Avvio addestramento StackingClassifier su TRAIN SET...')
print(f'[INFO] Train shape: {X_train_sel.shape}')
stacking_clf.fit(X_train_sel, y_train)
print('[INFO] Addestramento completato!')

# Salvataggio
joblib.dump(stacking_clf, 'stacking_pegi_model.pkl')
joblib.dump(sel_names, 'model_features.pkl')
print('[INFO] Modelli salvati con successo!')

In [ ]:
# Funzione di previsione PEGI basata sui tag inseriti
pegi_inv_map = {0: 3, 1: 7, 2: 12, 3: 16, 4: 18}

def tags_to_feature_vector(raw_tags):
    if isinstance(raw_tags, str):
        tags = clean_tags(raw_tags)
    elif isinstance(raw_tags, list):
        tags = [str(t).strip() for t in raw_tags if str(t).strip()]
    else:
        raise ValueError('I tag devono essere una stringa o una lista di tag')

    feature_dict = {name: 0 for name in sel_names}
    for tag in tags:
        feature_name = f"Tag_{tag.replace(' ','_').replace('-','_').replace('/','_')}"
        if feature_name in feature_dict:
            feature_dict[feature_name] = 1

    return pd.DataFrame([feature_dict], columns=sel_names)

def predict_pegi_from_tags(raw_tags):
    X_new = tags_to_feature_vector(raw_tags)
    proba = stacking_clf.predict_proba(X_new)[0]
    pred_class = int(proba.argmax())
    return {'predicted_class': pred_class, 'predicted_pegi': pegi_inv_map[pred_class], 'probabilities': proba}

# Esempio d'uso: gioco violento
example_tags_violent = ['Action', 'Violent', 'Gore', 'Blood', 'Mature', 'Shooter']
result_violent = predict_pegi_from_tags(example_tags_violent)
print('Test gioco violento:')
print(f'  Tags: {example_tags_violent}')
print(f'  Predicted PEGI: {result_violent["predicted_pegi"]}')
print(f'  Probabilities: {[f"{p:.3f}" for p in result_violent["probabilities"]]}')

print()

# Esempio d'uso: gioco per bambini
example_tags_kids = ['Casual', 'Puzzle', 'Family Friendly', 'Cute', 'Colorful']
result_kids = predict_pegi_from_tags(example_tags_kids)
print('Test gioco per bambini:')
print(f'  Tags: {example_tags_kids}')
print(f'  Predicted PEGI: {result_kids["predicted_pegi"]}')
print(f'  Probabilities: {[f"{p:.3f}" for p in result_kids["probabilities"]]}')

## Sezione 5: Valutazione Finale sul TEST SET

> **FIX CRITICO**: Tutte le metriche sono calcolate sul **test set** (dati MAI visti durante il training).
> Questo garantisce stime oneste e non gonfiate da data leakage.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    cohen_kappa_score, matthews_corrcoef,
    balanced_accuracy_score, roc_auc_score,
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize
import shap

print('\n[INFO] Sezione 5: Valutazione Finale sul TEST SET')
print('=' * 70)

# ============================================================
# FIX: Predizioni sul TEST SET (mai visto durante il training)
# ============================================================
y_pred = stacking_clf.predict(X_test_sel)
y_proba = stacking_clf.predict_proba(X_test_sel)
pegi_labels = ['PEGI 3', 'PEGI 7', 'PEGI 12', 'PEGI 16', 'PEGI 18']

print('\n1. CLASSIFICATION REPORT (TEST SET)\n')
print('=' * 70)
print(classification_report(y_test, y_pred, target_names=pegi_labels))
print('=' * 70)

In [ ]:
# Metriche avanzate sul TEST SET
kappa = cohen_kappa_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
bal_acc = balanced_accuracy_score(y_test, y_pred)

try:
    roc_auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
except Exception:
    roc_auc = float('nan')

print('\n2. METRICHE AVANZATE (TEST SET)\n')
print(f'  Cohen Kappa:          {kappa:.4f}')
print(f'  Matthews Corr. Coef: {mcc:.4f}')
print(f'  Balanced Accuracy:   {bal_acc:.4f}')
print(f'  Macro ROC-AUC:       {roc_auc:.4f}')
print()
print('  NOTA: Queste metriche sono calcolate sul TEST SET (hold-out 20%)')
print('  e rappresentano la performance reale del modello su dati non visti.')

In [ ]:
# Confusion Matrix sul TEST SET
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

print('\n3. MATRICE DI CONFUSIONE (TEST SET)\n')
plt.figure(figsize=(10, 8))
sns.heatmap(cm_norm, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=pegi_labels, yticklabels=pegi_labels,
            cbar_kws={'label': 'Frequenza (%)'})
plt.title('Matrice di Confusione Normalizzata - TEST SET (%)', fontsize=14, fontweight='bold')
plt.ylabel('Classe Reale', fontsize=12)
plt.xlabel('Classe Predetta', fontsize=12)
plt.tight_layout()
plt.show()

# Stampa anche i conteggi assoluti
print('\nConteggi assoluti:')
cm_df = pd.DataFrame(cm, index=pegi_labels, columns=pegi_labels)
print(cm_df)

In [ ]:
# ROC Curves sul TEST SET
y_bin = label_binarize(y_test, classes=[0, 1, 2, 3, 4])
colors_roc = ['steelblue', 'darkorange', 'green', 'crimson', 'purple']

print('\n4. CURVE ROC MULTI-CLASSE (TEST SET)\n')
plt.figure(figsize=(10, 7))
for i, (lab, col) in enumerate(zip(pegi_labels, colors_roc)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
    roc_i = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=col, lw=2, label=f'{lab} (AUC = {roc_i:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve Multi-Classe - TEST SET', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.35)
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance (dal modello XGBoost nello stacking)
imp_scores = stacking_clf.named_estimators_['xgb'].feature_importances_
imp_df = pd.DataFrame({'Feature': sel_names, 'Importance': imp_scores})
imp_df = imp_df.sort_values('Importance', ascending=False)

print('\n5. TOP-25 FEATURE IMPORTANCE\n')
top_n = min(25, len(imp_df))
print(imp_df.head(top_n).to_string(index=False))

plt.figure(figsize=(13, 9))
sns.barplot(data=imp_df.head(top_n), x='Importance', y='Feature',
            palette='viridis', hue='Feature', legend=False)
plt.title(f'Top-{top_n} Feature Importance (XGBoost)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Analysis (sul TEST set per interpretabilita' onesta)
print('\n6. SHAP VALUE ANALYSIS\n')
explainer = shap.TreeExplainer(stacking_clf.named_estimators_['xgb'])
sample_size = min(300, len(X_test_sel))
X_sample = X_test_sel.sample(n=sample_size, random_state=42)
shap_values = explainer.shap_values(X_sample)

plt.figure()
shap.summary_plot(shap_values, X_sample, plot_type='bar',
                  class_names=pegi_labels, show=False)
plt.title('SHAP Feature Importance per Classe (TEST SET)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP detail per PEGI 18 (classe 4) - per capire cosa guida le predizioni sui giochi violenti
print('\n7. SHAP DETAIL PER PEGI 18 (GIOCHI VIOLENTI)\n')
if isinstance(shap_values, list) and len(shap_values) > 4:
    plt.figure()
    shap.summary_plot(shap_values[4], X_sample, show=False, max_display=20)
    plt.title('SHAP Values - PEGI 18 (Giochi Violenti)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('[INFO] SHAP values per classe singola non disponibili in questo formato')

In [ ]:
# Riepilogo finale
print('\n' + '=' * 70)
print('RIEPILOGO FINALE')
print('=' * 70)
print(f'\nDataset: {len(df_final)} campioni ({len(X_train_sel)} train / {len(X_test_sel)} test)')
print(f'Feature: {len(sel_names)} tag selezionati (su {X_tags.shape[1]} totali)')
print(f'\nMetriche CV (train, stima onesta):')
print(f'  XGBoost:    F1 Macro = {study_xgb.best_value:.4f}')
print(f'  LightGBM:   F1 Macro = {study_lgb.best_value:.4f}')
print(f'  CatBoost:   F1 Macro = {study_cat.best_value:.4f}')
print(f'  ExtraTrees: F1 Macro = {study_et.best_value:.4f}')
print(f'\nMetriche TEST SET (hold-out 20%, mai visto):')
print(f'  Balanced Accuracy: {bal_acc:.4f}')
print(f'  Cohen Kappa:       {kappa:.4f}')
print(f'  MCC:               {mcc:.4f}')
print(f'  ROC-AUC Macro:     {roc_auc:.4f}')
print(f'\nFix applicati:')
print(f'  [x] Train/test split stratificato prima di feature selection')
print(f'  [x] Feature selection solo su train')
print(f'  [x] class_weight=balanced (no SMOTE su feature binarie)')
print(f'  [x] Tuning Optuna per tutti e 4 i modelli')
print(f'  [x] Valutazione finale solo su test set')
print(f'  [x] Mappatura PEGI estesa (recupero rating non standard)')
print('=' * 70)